In [12]:
import re
import pandas as pd
from tqdm import tqdm
from transformers import T5Tokenizer

In [13]:
INPUT_CSV = "../artifacts/data_ingestion/arxiv_dataset/train.csv"
OUTPUT_CSV = "final_dataset.csv"

MODEL_NAME = "t5-small"
MAX_INPUT_TOKENS = 512
CHUNK_SIZE = 400
OVERLAP = 80

tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)

In [15]:
def clean_text(text: str) -> str:
    text = re.sub(r'@xcite', '', text)
    text = re.sub(r'@xmath\d+', '', text)

    text = text.replace("\n", " ")

    text = re.sub(r'\s+,', ',', text)
    text = re.sub(r'\s+\.', '.', text)
    text = re.sub(r'\s*-\s*', '-', text)

    text = re.sub(r'\s+', ' ', text)
    return text.strip().lower()

In [16]:
def chunk_text(text):
    tokens = tokenizer.encode(
        text,
        add_special_tokens=False,
        truncation=False  # 🔥 IMPORTANT FIX
    )

    chunks = []
    start = 0

    while start < len(tokens):
        end = start + CHUNK_SIZE
        chunk_tokens = tokens[start:end]

        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append(chunk_text)

        start += CHUNK_SIZE - OVERLAP

    return chunks

In [17]:
def preprocess(input_path, output_path):
    df = pd.read_csv(input_path)

    inputs = []
    targets = []

    for _, row in tqdm(df.iterrows(), total=len(df)):
        raw_text = str(row["text"])
        raw_summary = str(row["summary"])

        #CLEAN
        clean_txt = clean_text(raw_text)
        clean_sum = clean_text(raw_summary)

        if len(clean_txt) < 50:
            continue

        #CHUNK
        chunks = chunk_text(clean_txt)

        #FORMAT FOR T5
        for chunk in chunks:
            inputs.append(f"summarize: {chunk}")
            targets.append(clean_sum)  # simple approach

    final_df = pd.DataFrame({
        "input_text": inputs,
        "target_text": targets
    })

    final_df.to_csv(output_path, index=False)

    print(f"✅ Done! Final dataset size: {len(final_df)}")

In [ ]:
 preprocess(INPUT_CSV, OUTPUT_CSV)